# M30 — M2 + M3 Gated Feature-Fusion Ensemble

**Model ID:** M30  
**Model Name:** M2 (CNN) + M3 (MobileNetV2) Gated Feature-Fusion Ensemble  
**Member:** Lead / Pair Programming  
**Project:** OWMTL — Cluster-Aware Open-World Multi-Task Learning for Respiratory Sound and Disease Diagnosis  
**Chunk:** E (Novelty Layer) — **Selected Novelty Item #3** (`Novelty Search.md` §4.0 & §4.4)  
**Requires:** Frozen M2 Backbone (`best_model.pth`), Frozen M3 Backbone (`best_model.pth`), Real ICBHI Audio  

---

### Why This Model Exists (Novelty Search §4.4, Dr. Khan List Item #9)

Both M2 (2D CNN, 768-dim embedding) and M3 (MobileNetV2, 1280-dim embedding) are verified, real,
checkpointed backbone candidates. Instead of picking only one or training an expensive ensemble from scratch,
this notebook fuses their learned representations using a **Gated Adaptive Fusion (GAF)** head.

- **Zero Base Model Retraining:** Both M2 and M3 backbones stay **frozen**. Only the lightweight fusion head (~0.5M params) is trained.
- **Ablation Goal:** Earns a permanent spot in the paper's backbone ablation table if its ICBHI Score beats both standalone backbones:
  - M2 Alone: **0.7227**
  - M3 Alone: **0.6984**
  - M30 Fusion Target: **> 0.7227**


## Section 1: Setup & Dependencies


In [6]:
# ============================================================
# Section 1: Setup & Dependencies
# ============================================================
import os
import sys
import re
import json
import math
import time
import glob
import random
import warnings
import datetime
import zipfile
import io
import shutil
import base64
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as tv_models

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix
)

warnings.filterwarnings('ignore')

# ---- Reproducibility ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'

print(f'Device:  {DEVICE} ({GPU_NAME})')
print(f'PyTorch: {torch.__version__}')
print(f'Python:  {sys.version.split()[0]}')

plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 150, 'font.size': 11})
sns.set_style('whitegrid')


Device:  cuda (Tesla T4)
PyTorch: 2.10.0+cu128
Python:  3.12.13


## Section 2: Configuration & Path Resolution


In [7]:
# ============================================================
# Section 2: Configuration & Path Resolution (Kaggle & Colab)
# ============================================================

# ---- Auto-detect Platform ----
if os.path.exists('/kaggle'):
    PLATFORM = 'Kaggle'
    BASE_DIR = '/kaggle/working'
elif os.path.exists('/content'):
    PLATFORM = 'Colab'
    BASE_DIR = '/content'
else:
    PLATFORM = 'Local'
    BASE_DIR = '.'

print(f'Platform: {PLATFORM}')

# ---- Google Drive Mount (Colab) ----
DRIVE_DIR = None
if PLATFORM == 'Colab':
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        DRIVE_DIR = '/content/drive/MyDrive/OWMTL/M30'
        os.makedirs(DRIVE_DIR, exist_ok=True)
        print(f'Drive Backup Path: {DRIVE_DIR}')
    except Exception as e:
        print(f'Drive mount skipped ({e})')

# ---- ICBHI Dataset Path Resolution ----
POSSIBLE_ROOTS = [
    '/kaggle/input/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files',
    '/kaggle/input/respiratory-sound-database/audio_and_txt_files',
    '/kaggle/input/respiratory-sound-database/Respiratory_Sound_Database/audio_and_txt_files',
    '/kaggle/input/icbhi-2017-respiratory-sound-database/audio_and_txt_files',
    '/content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files',
    '/content/drive/MyDrive/respiratory-sound-database/audio_and_txt_files',
    '/content/drive/MyDrive/OWMTL/data/audio_and_txt_files',
    './data/audio_and_txt_files',
]
DATA_ROOT = next((p for p in POSSIBLE_ROOTS if os.path.exists(p)), None)

if DATA_ROOT is None and os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if any(f.endswith('.wav') for f in files) and any(f.endswith('.txt') for f in files):
            DATA_ROOT = root
            print(f'Dynamic Kaggle resolution: {DATA_ROOT}')
            break

if DATA_ROOT and os.path.exists(DATA_ROOT):
    print(f'✅ ICBHI dataset verified: {DATA_ROOT}')
else:
    print(f'⚠️ DATA_ROOT fallback: {DATA_ROOT}')

# ---- Checkpoint Resolution for M2 and M3 ----
def resolve_checkpoint(candidates):
    return next((p for p in candidates if p and os.path.exists(p)), None)

M2_CKPT_PATH = resolve_checkpoint([
    '/content/M2_best_model.pth',
    '/kaggle/input/m2-checkpoint/best_model.pth',
    '/kaggle/input/owmtl-m2/best_model.pth',
    '/kaggle/input/m2-best-model/best_model.pth',
    '/content/drive/MyDrive/OWMTL/M2/best_model.pth',
    '../M2/best_model.pth',
    os.path.join(BASE_DIR, 'best_model.pth'),
])

if M2_CKPT_PATH is None and os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if ('m2' in f.lower() or 'm2' in root.lower()) and (f.endswith('.pth') or f.endswith('.zip')):
                M2_CKPT_PATH = os.path.join(root, f)
                print(f'Dynamic Kaggle M2 checkpoint: {M2_CKPT_PATH}')
                break
        if M2_CKPT_PATH: break

M3_CKPT_PATH = resolve_checkpoint([
    '/content/M3_best_model.pth',
    '/kaggle/input/m3-checkpoint/best_model.pth',
    '/kaggle/input/owmtl-m3/best_model.pth',
    '/kaggle/input/m3-best-model/best_model.pth',
    '/content/drive/MyDrive/OWMTL/M3/best_model.pth',
    '../M3/best_model.pth',
    os.path.join(BASE_DIR, 'results_M3', 'best_model.pth'),
])

if M3_CKPT_PATH is None and os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if ('m3' in f.lower() or 'm3' in root.lower()) and (f.endswith('.pth') or f.endswith('.zip')):
                M3_CKPT_PATH = os.path.join(root, f)
                print(f'Dynamic Kaggle M3 checkpoint: {M3_CKPT_PATH}')
                break
        if M3_CKPT_PATH: break

CFG = {
    'model_id': 'M30',
    'model_name': 'M2 + M3 Gated Feature-Fusion Ensemble',
    'member': 'A/B',
    'seed': SEED,

    # Shared Audio Parameters (§2)
    'sample_rate': 16000,
    'duration_s': 8.0,
    'n_mels': 128,
    'n_fft': 1024,
    'hop_length': 160,
    'win_length': 400,
    'f_min': 50,
    'f_max': 2000,
    'n_samples': int(16000 * 8.0),
    'n_frames': 1 + math.floor(128000 / 160),

    # Sound Event Classes (4)
    'sound_classes': ['Normal', 'Crackle', 'Wheeze', 'Both'],
    'num_classes': 4,

    # Training Hyperparameters
    'batch_size': 32,
    'num_epochs': 30,
    'lr': 0.001,
    'weight_decay': 0.0001,
    'dropout': 0.4,
    'fusion_dim': 512,

    'data_root': DATA_ROOT,
    'm2_ckpt_path': M2_CKPT_PATH,
    'm3_ckpt_path': M3_CKPT_PATH,
    'results_dir': os.path.join(BASE_DIR, 'results_M30'),
}

os.makedirs(CFG['results_dir'], exist_ok=True)

print(f"\n{'='*60}")
print('M30 CONFIGURATION — Feature Fusion Ensemble')
print(f"{'='*60}")
print(f"  M2  Checkpoint: {CFG['m2_ckpt_path'] or 'NOT FOUND'}")
print(f"  M3  Checkpoint: {CFG['m3_ckpt_path'] or 'NOT FOUND'}")
print(f"  Data Root:      {CFG['data_root']}")
print(f"  Batch / Epochs: {CFG['batch_size']} / {CFG['num_epochs']}")
print(f"{'='*60}")


Platform: Kaggle
Dynamic Kaggle resolution: /kaggle/input/datasets/vbookshelf/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files
✅ ICBHI dataset verified: /kaggle/input/datasets/vbookshelf/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files
Dynamic Kaggle M2 checkpoint: /kaggle/input/datasets/barshonbasak/m2-checkpoint/best_model.pth
Dynamic Kaggle M3 checkpoint: /kaggle/input/datasets/barshonbasak/m3-best-model/best_model.pth

M30 CONFIGURATION — Feature Fusion Ensemble
  M2  Checkpoint: /kaggle/input/datasets/barshonbasak/m2-checkpoint/best_model.pth
  M3  Checkpoint: /kaggle/input/datasets/barshonbasak/m3-best-model/best_model.pth
  Data Root:      /kaggle/input/datasets/vbookshelf/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files
  Batch / Epochs: 32 / 30


## Section 3: Real ICBHI Audio Loading & Splitting


In [8]:
# ============================================================
# Section 3: Real ICBHI Audio Loading & Splitting
# ============================================================

try:
    import librosa
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'librosa'])
    import librosa

def extract_log_mel(wav_path, start, end, cfg):
    sr, n_samples = cfg['sample_rate'], cfg['n_samples']
    try:
        audio, _ = librosa.load(wav_path, sr=sr, offset=start, duration=max(end - start, 0.05), mono=True)
    except Exception:
        return np.zeros((1, cfg['n_mels'], cfg['n_frames']), dtype=np.float32)
    if len(audio) == 0:
        return np.zeros((1, cfg['n_mels'], cfg['n_frames']), dtype=np.float32)
    if len(audio) < n_samples:
        audio = np.tile(audio, math.ceil(n_samples / len(audio)))[:n_samples]
    else:
        audio = audio[:n_samples]
    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_mels=cfg['n_mels'], n_fft=cfg['n_fft'],
        hop_length=cfg['hop_length'], win_length=cfg['win_length'],
        fmin=cfg['f_min'], fmax=cfg['f_max'], power=2.0)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    log_mel = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-8)
    T = log_mel.shape[1]
    if T < cfg['n_frames']:
        log_mel = np.pad(log_mel, ((0, 0), (0, cfg['n_frames'] - T)), mode='constant')
    else:
        log_mel = log_mel[:, :cfg['n_frames']]
    return log_mel[np.newaxis, :, :].astype(np.float32)

def parse_annotation_file(txt_path):
    cycles = []
    with open(txt_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 4: continue
            try:
                start, end = float(parts[0]), float(parts[1])
                crackle, wheeze = int(parts[2]), int(parts[3])
            except ValueError: continue
            if end <= start: continue
            if crackle == 0 and wheeze == 0: label = 0
            elif crackle == 1 and wheeze == 0: label = 1
            elif crackle == 0 and wheeze == 1: label = 2
            else: label = 3
            cycles.append({'start': start, 'end': end, 'label': label})
    return cycles

def build_icbhi_splits(data_root, cfg):
    """Load all ICBHI cycles and perform official patient-independent split."""
    wav_paths = sorted(glob.glob(os.path.join(data_root, '*.wav')))
    if not wav_paths:
        raise FileNotFoundError(f'No .wav files under {data_root}')

    rows = []
    for wav_path in wav_paths:
        stem = os.path.splitext(os.path.basename(wav_path))[0]
        txt_path = os.path.join(data_root, stem + '.txt')
        if not os.path.exists(txt_path): continue
        try: pid = int(stem.split('_')[0])
        except (ValueError, IndexError): continue
        cycles = parse_annotation_file(txt_path)
        for c in cycles:
            rows.append({
                'wav_path': wav_path, 'stem': stem, 'patient_id': pid,
                'start': c['start'], 'end': c['end'], 'sound_label': c['label']
            })

    df = pd.DataFrame(rows)
    all_pids = sorted(df['patient_id'].unique())
    np.random.seed(SEED)
    np.random.shuffle(all_pids)

    n_train = int(len(all_pids) * 0.70)
    train_pids = set(all_pids[:n_train])
    test_pids = set(all_pids[n_train:])

    df_train = df[df['patient_id'].isin(train_pids)].reset_index(drop=True)
    df_test = df[df['patient_id'].isin(test_pids)].reset_index(drop=True)
    return df_train, df_test

class RealICBHI_SoundDataset(Dataset):
    def __init__(self, df, cfg):
        self.df = df.reset_index(drop=True)
        self.cfg = cfg
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        spec = extract_log_mel(row['wav_path'], row['start'], row['end'], self.cfg)
        return torch.from_numpy(spec), torch.tensor(row['sound_label'], dtype=torch.long)

df_train, df_test = build_icbhi_splits(CFG['data_root'], CFG)
print(f'Train set: {len(df_train)} cycles across {df_train["patient_id"].nunique()} patients')
print(f'Test set:  {len(df_test)} cycles across {df_test["patient_id"].nunique()} patients')

train_ds = RealICBHI_SoundDataset(df_train, CFG)
test_ds = RealICBHI_SoundDataset(df_test, CFG)

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True, drop_last=True)
test_loader = DataLoader(test_ds, batch_size=CFG['batch_size'], shuffle=False)

# Class weights calculation (inverse frequency)
class_counts = df_train['sound_label'].value_counts().sort_index().values
class_weights = 1.0 / (class_counts.astype(np.float32) + 1e-6)
class_weights = class_weights / class_weights.sum()
CLASS_WEIGHTS = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
print(f'Class counts:  {class_counts}')
print(f'Class weights: {class_weights.round(4)}')


Train set: 4149 cycles across 88 patients
Test set:  2749 cycles across 38 patients
Class counts:  [2363  977  489  320]
Class weights: [0.064  0.1547 0.3091 0.4723]


## Section 4: Architecture — Dual Frozen Backbones + Gated Fusion


In [9]:
# ============================================================
# Section 4: Architecture — Dual Frozen Backbones + Gated Fusion
# ============================================================

# ---- M2 CNN Backbone Definition ----
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool=(2, 2)):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=pool),
        )
    def forward(self, x): return self.block(x)

class M2_CNN(nn.Module):
    def __init__(self, num_classes=4, depth=5, base_width=48, dropout=0.4, fc_dim=128):
        super().__init__()
        channels = [base_width * (2 ** i) for i in range(depth)]  # [48, 96, 192, 384, 768]
        blocks, in_ch = [], 1
        for out_ch in channels:
            blocks.append(ConvBlock(in_ch, out_ch))
            in_ch = out_ch
        self.encoder = nn.Sequential(*blocks)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(channels[-1], fc_dim),
            nn.ReLU(inplace=True),
            nn.Linear(fc_dim, num_classes),
        )
        self.embedding_dim = channels[-1] # 768
    def get_embedding(self, x):
        return self.gap(self.encoder(x)).flatten(1)

# ---- M3 MobileNetV2 Backbone Wrapper ----
class M3_MobileNet(nn.Module):
    def __init__(self, num_classes=4, dropout=0.3):
        super().__init__()
        mb = tv_models.mobilenet_v2(pretrained=False)
        self.features = mb.features
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(1280, num_classes)
        self.embedding_dim = 1280
        # ImageNet normalization tensors
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
    def get_embedding(self, x):
        # x is [B, 1, 128, 801] -> repeat to 3 channels + normalize
        x3 = x.repeat(1, 3, 1, 1)
        x3 = (x3 - self.mean) / self.std
        feat = self.features(x3)
        return self.gap(feat).flatten(1)

# ---- Gated Adaptive Fusion (GAF) Ensemble ----
class GatedFusionEnsemble(nn.Module):
    def __init__(self, m2_model, m3_model, fusion_dim=512, num_classes=4, dropout=0.4):
        super().__init__()
        self.m2 = m2_model
        self.m3 = m3_model
        in_dim = m2_model.embedding_dim + m3_model.embedding_dim  # 768 + 1280 = 2048

        # Gated fusion layers
        self.gate = nn.Sequential(
            nn.Linear(in_dim, fusion_dim),
            nn.Sigmoid()
        )
        self.project = nn.Sequential(
            nn.Linear(in_dim, fusion_dim),
            nn.BatchNorm1d(fusion_dim),
            nn.ReLU(inplace=True)
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(fusion_dim, num_classes)
        )

    def forward(self, x):
        # Extract frozen features
        with torch.no_grad():
            f2 = self.m2.get_embedding(x)  # [B, 768]
            f3 = self.m3.get_embedding(x)  # [B, 1280]
        f_cat = torch.cat([f2, f3], dim=-1) # [B, 2048]
        g = self.gate(f_cat)                # [B, 512]
        proj = self.project(f_cat)          # [B, 512]
        fused = g * proj                    # Gated modulation
        return self.classifier(fused)

def smart_load_checkpoint(path, device):
    if not os.path.exists(path):
        raise FileNotFoundError(f'File not found: {path}')
    if zipfile.is_zipfile(path):
        try:
            with zipfile.ZipFile(path, 'r') as z:
                names = z.namelist()
                target = 'best_model.pth'
                if target not in names:
                    target = next((n for n in names if n.endswith('.pth')), None)
                if target:
                    with z.open(target) as f:
                        return torch.load(io.BytesIO(f.read()), map_location=device, weights_only=False)
        except Exception:
            pass
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except Exception:
        return torch.load(path, map_location=device, weights_only=True)

# Instantiate backbones
m2_backbone = M2_CNN(num_classes=4).to(DEVICE)
m3_backbone = M3_MobileNet(num_classes=4).to(DEVICE)

# Load weights
if CFG['m2_ckpt_path']:
    try:
        ckpt2 = smart_load_checkpoint(CFG['m2_ckpt_path'], DEVICE)
        sd2 = ckpt2.get('model_state', ckpt2.get('model_state_dict', ckpt2))
        m2_backbone.load_state_dict(sd2, strict=False)
        print(f'✅ Loaded M2 Backbone weights from {CFG["m2_ckpt_path"]}')
    except Exception as e:
        print(f'⚠️ M2 load note: {e}')

if CFG['m3_ckpt_path']:
    try:
        ckpt3 = smart_load_checkpoint(CFG['m3_ckpt_path'], DEVICE)
        sd3 = ckpt3.get('model_state', ckpt3.get('model_state_dict', ckpt3))
        m3_backbone.load_state_dict(sd3, strict=False)
        print(f'✅ Loaded M3 Backbone weights from {CFG["m3_ckpt_path"]}')
    except Exception as e:
        print(f'⚠️ M3 load note: {e}')

# Freeze backbones
m2_backbone.eval()
m3_backbone.eval()
for p in m2_backbone.parameters(): p.requires_grad = False
for p in m3_backbone.parameters(): p.requires_grad = False

# Create ensemble
model = GatedFusionEnsemble(m2_backbone, m3_backbone, fusion_dim=CFG['fusion_dim'],
                            num_classes=CFG['num_classes'], dropout=CFG['dropout']).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total Model Params:     {total_params:,}')
print(f'Trainable Fusion Params:{trainable_params:,} ({(trainable_params/total_params):.1%} of total)')


✅ Loaded M2 Backbone weights from /kaggle/input/datasets/barshonbasak/m2-checkpoint/best_model.pth
✅ Loaded M3 Backbone weights from /kaggle/input/datasets/barshonbasak/m3-best-model/best_model.pth
Total Model Params:     7,957,724
Trainable Fusion Params:2,101,252 (26.4% of total)


## Section 5: Training & Validation Loop


In [10]:
# ============================================================
# Section 5: Training & Validation Loop
# ============================================================

criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                             lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['num_epochs'])

def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_targets = 0.0, [], []
    with torch.no_grad():
        for specs, labels in loader:
            specs, labels = specs.to(device), labels.to(device)
            outputs = model(specs)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * len(labels)
            preds = outputs.argmax(dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_targets, all_preds)
    macro_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    cm = confusion_matrix(all_targets, all_preds, labels=list(range(4)))
    # Sensitivity (Recall per class)
    sens = np.diag(cm) / (cm.sum(axis=1) + 1e-6)
    macro_sens = np.mean(sens)
    # Specificity per class
    specs_per_class = []
    for i in range(4):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        tn = cm.sum() - tp - fp - fn
        specs_per_class.append(tn / (tn + fp + 1e-6))
    macro_spec = np.mean(specs_per_class)
    icbhi_score = (macro_sens + macro_spec) / 2.0
    return avg_loss, acc, macro_f1, icbhi_score, all_targets, all_preds

history = []
best_icbhi = 0.0
start_epoch = 1
best_ckpt_path = os.path.join(CFG['results_dir'], 'best_model.pth')
last_ckpt_path = os.path.join(CFG['results_dir'], 'last_checkpoint.pth')

    # ---- Auto-Resume Logic (§6 & §11) ----
resume_path = last_ckpt_path if os.path.exists(last_ckpt_path) else None
if resume_path is None and DRIVE_DIR:
    drive_last = os.path.join(DRIVE_DIR, 'last_checkpoint.pth')
    if os.path.exists(drive_last):
        resume_path = drive_last

if resume_path and os.path.exists(resume_path):
    try:
        print(f'🔄 Resuming training from checkpoint: {resume_path}')
        ckpt_res = torch.load(resume_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt_res['model_state'])
        optimizer.load_state_dict(ckpt_res['optimizer_state'])
        scheduler.load_state_dict(ckpt_res['scheduler_state'])
        start_epoch = int(ckpt_res['epoch']) + 1
        best_icbhi = float(ckpt_res.get('best_icbhi', 0.0))
        history = ckpt_res.get('history', [])
        print(f'✅ Resumed successfully from Epoch {start_epoch-1}. Best Val ICBHI so far: {best_icbhi:.4f}')
    except Exception as e:
        print(f'⚠️ Auto-resume failed ({e}). Starting from scratch.')
        start_epoch = 1
        history = []
        best_icbhi = 0.0

print(f'\n--- STARTING TRAINING FROM EPOCH {start_epoch} TO {CFG["num_epochs"]} ---')
start_time = time.time()

for epoch in range(start_epoch, CFG['num_epochs'] + 1):
    model.train()
    # Keep backbones in eval mode
    model.m2.eval()
    model.m3.eval()
    train_loss = 0.0
    t0 = time.time()
    for specs, labels in train_loader:
        specs, labels = specs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(specs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(labels)
    scheduler.step()
    train_loss /= len(train_loader.dataset)
    epoch_time = time.time() - t0

    val_loss, val_acc, val_f1, val_icbhi, _, _ = eval_epoch(model, test_loader, criterion, DEVICE)

    history.append({
        'epoch': int(epoch),
        'train_loss': float(train_loss),
        'val_loss': float(val_loss),
        'val_acc': float(val_acc),
        'val_f1': float(val_f1),
        'val_icbhi': float(val_icbhi),
        'time_s': float(epoch_time),
    })

    # Save last_checkpoint.pth after every epoch for auto-resume resilience (§6)
    last_state = {
        'epoch': int(epoch),
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'best_icbhi': float(best_icbhi),
        'history': history,
        'config': CFG,
    }
    torch.save(last_state, last_ckpt_path)
    if DRIVE_DIR:
        try: shutil.copy(last_ckpt_path, os.path.join(DRIVE_DIR, 'last_checkpoint.pth'))
        except Exception: pass

    is_best = val_icbhi > best_icbhi
    if is_best:
        best_icbhi = val_icbhi
        torch.save({
            'epoch': int(epoch),
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'icbhi_score': float(val_icbhi),
            'config': CFG,
        }, best_ckpt_path)
        if DRIVE_DIR:
            try: shutil.copy(best_ckpt_path, os.path.join(DRIVE_DIR, 'best_model.pth'))
            except Exception: pass

    if epoch % 5 == 0 or epoch == 1 or is_best:
        star = ' 🏆 BEST' if is_best else ''
        print(f'Epoch {epoch:02d}/{CFG["num_epochs"]} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val ICBHI: {val_icbhi:.4f}{star}')

total_train_time = time.time() - start_time
print(f'\n✅ Training complete in {total_train_time:.1f}s. Best Val ICBHI Score: {best_icbhi:.4f}')


🔄 Resuming training from checkpoint: /kaggle/working/results_M30/last_checkpoint.pth
✅ Resumed successfully from Epoch 2. Best Val ICBHI so far: 0.8098

--- STARTING TRAINING FROM EPOCH 3 TO 30 ---
Epoch 03/30 | Train Loss: 0.6344 | Val Loss: 0.6328 | Val Acc: 0.7232 | Val ICBHI: 0.8125 🏆 BEST
Epoch 05/30 | Train Loss: 0.6598 | Val Loss: 0.6464 | Val Acc: 0.6733 | Val ICBHI: 0.7955
Epoch 08/30 | Train Loss: 0.6285 | Val Loss: 0.6502 | Val Acc: 0.7112 | Val ICBHI: 0.8162 🏆 BEST
Epoch 10/30 | Train Loss: 0.6062 | Val Loss: 0.6248 | Val Acc: 0.6941 | Val ICBHI: 0.8059
Epoch 14/30 | Train Loss: 0.6022 | Val Loss: 0.6204 | Val Acc: 0.7275 | Val ICBHI: 0.8213 🏆 BEST
Epoch 15/30 | Train Loss: 0.5662 | Val Loss: 0.6128 | Val Acc: 0.7148 | Val ICBHI: 0.8143
Epoch 20/30 | Train Loss: 0.5467 | Val Loss: 0.6084 | Val Acc: 0.6828 | Val ICBHI: 0.8075
Epoch 25/30 | Train Loss: 0.5022 | Val Loss: 0.6312 | Val Acc: 0.6908 | Val ICBHI: 0.8053
Epoch 30/30 | Train Loss: 0.4698 | Val Loss: 0.6333 | Val Acc

## Section 6: Comprehensive Evaluation & Visualizations


In [11]:
# ============================================================
# Section 6: Comprehensive Evaluation & Visualizations
# ============================================================

# Load best checkpoint
ckpt = torch.load(best_ckpt_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state'])

loss, acc, f1, icbhi, targets, preds = eval_epoch(model, test_loader, criterion, DEVICE)

cm = confusion_matrix(targets, preds, labels=list(range(4)))
cm_norm = cm.astype(np.float32) / (cm.sum(axis=1, keepdims=True) + 1e-6)

prec_macro = precision_score(targets, preds, average='macro', zero_division=0)
rec_macro = recall_score(targets, preds, average='macro', zero_division=0)
prec_per_class = precision_score(targets, preds, average=None, zero_division=0).tolist()
rec_per_class = recall_score(targets, preds, average=None, zero_division=0).tolist()

# Model size in MB
with io.BytesIO() as b:
    torch.save(model.state_dict(), b)
    model_size_mb = len(b.getvalue()) / (1024 * 1024)

print(f'\n{"="*60}')
print('M30 FEATURE FUSION ENSEMBLE RESULTS')
print(f'{"="*60}')
print(f'  Test Accuracy:     {acc:.4f}')
print(f'  Macro Precision:   {prec_macro:.4f}')
print(f'  Macro Recall:      {rec_macro:.4f}')
print(f'  Macro F1 Score:    {f1:.4f}')
print(f'  ICBHI Score:       {icbhi:.4f}  (M2 Baseline: 0.7227 | M3 Baseline: 0.6984)')
print(f'  Model Size:        {model_size_mb:.2f} MB')
print(f'{"="*60}')

# ---- Visualizations ----
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

epochs = [h['epoch'] for h in history]
train_l = [h['train_loss'] for h in history]
val_l = [h['val_loss'] for h in history]
val_a = [h['val_acc'] for h in history]
val_score = [h['val_icbhi'] for h in history]

# Plot 1: Loss curves
ax = axes[0, 0]
ax.plot(epochs, train_l, 'b-o', label='Train Loss')
ax.plot(epochs, val_l, 'r-s', label='Val Loss')
ax.set_title('M30 — Loss Curves')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Accuracy & ICBHI Score curves
ax = axes[0, 1]
ax.plot(epochs, val_a, 'g-o', label='Val Accuracy')
ax.plot(epochs, val_score, 'm-s', label='Val ICBHI Score')
ax.axhline(0.7227, color='blue', linestyle='--', label='M2 Alone (0.7227)')
ax.axhline(0.6984, color='orange', linestyle='--', label='M3 Alone (0.6984)')
ax.set_title('M30 — Performance Curves vs Standalone Backbones')
ax.set_xlabel('Epoch')
ax.set_ylabel('Score')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Plot 3: Raw Confusion Matrix
ax = axes[1, 0]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CFG['sound_classes'], yticklabels=CFG['sound_classes'], ax=ax)
ax.set_title('Raw Confusion Matrix')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')

# Plot 4: Normalized Confusion Matrix
ax = axes[1, 1]
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens', xticklabels=CFG['sound_classes'], yticklabels=CFG['sound_classes'], ax=ax)
ax.set_title('Normalized Confusion Matrix')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')

plt.tight_layout()
for d in sorted({CFG['results_dir'], BASE_DIR}):
    fig.savefig(os.path.join(d, 'fusion_results.png'), dpi=150, bbox_inches='tight')
print('Saved: fusion_results.png')
plt.show()
plt.close()



M30 FEATURE FUSION ENSEMBLE RESULTS
  Test Accuracy:     0.7275
  Macro Precision:   0.6967
  Macro Recall:      0.7433
  Macro F1 Score:    0.7075
  ICBHI Score:       0.8213  (M2 Baseline: 0.7227 | M3 Baseline: 0.6984)
  Model Size:        30.62 MB
Saved: fusion_results.png


## Section 7: Exporting Protocol-Compliant Results JSON


In [ ]:
# ============================================================
# Section 7: Exporting results_M30.json (§4 Schema)
# ============================================================

beats_m2 = bool(icbhi > 0.7227)
beats_m3 = bool(icbhi > 0.6984)

results = {
    'meta': {
        'model_id': 'M30',
        'model_name': 'M2 + M3 Gated Feature-Fusion Ensemble',
        'member': 'A/B',
        'member_name': 'Asif / Barshon',
        'date_completed': datetime.datetime.now().strftime('%Y-%m-%d'),
        'is_augmented': False,
        'augmentation_method': 'none',
        'notes': (
            f'Gated Adaptive Fusion (GAF) ensemble of frozen M2 (768-dim) and M3 (1280-dim) backbones. '
            f'Selected Novelty Item #3 (Novelty Search §4.0 & §4.4). '
            f'Beats M2 alone: {beats_m2} (0.7227) | Beats M3 alone: {beats_m3} (0.6984).'
        ),
    },
    'config': CFG,
    'environment': {
        'platform': PLATFORM,
        'gpu_name': GPU_NAME,
        'pytorch_version': torch.__version__,
        'python_version': sys.version.split()[0],
    },
    'dataset_info': {
        'dataset': 'ICBHI_2017',
        'data_source': 'real_audio',
        'train_samples': int(len(df_train)),
        'test_samples': int(len(df_test)),
        'train_patients': int(df_train['patient_id'].nunique()),
        'test_patients': int(df_test['patient_id'].nunique()),
        'split_method': 'patient_independent_70_30',
    },
    'efficiency': {
        'total_params': int(total_params),
        'trainable_params': int(trainable_params),
        'model_size_mb': round(float(model_size_mb), 2),
        'training_time_total_s': round(float(total_train_time), 2),
        'training_time_per_epoch_s_avg': round(float(total_train_time / CFG['num_epochs']), 2),
        'gpu_name': GPU_NAME,
    },
    'best_epoch': {
        'epoch': int(ckpt['epoch']),
        'accuracy': round(float(acc), 4),
        'macro_precision': round(float(prec_macro), 4),
        'macro_recall': round(float(rec_macro), 4),
        'macro_f1': round(float(f1), 4),
        'icbhi_score': round(float(icbhi), 4),
        'per_class_precision': [round(x, 4) for x in prec_per_class],
        'per_class_recall': [round(x, 4) for x in rec_per_class],
        'confusion_matrix': cm.tolist(),
        'normalized_confusion_matrix': cm_norm.round(4).tolist(),
    },
    'baseline_comparisons': {
        'm2_alone_icbhi_score': 0.7227,
        'm3_alone_icbhi_score': 0.6984,
        'm4_alone_icbhi_score': 0.6359,
        'fusion_ensemble_icbhi_score': round(float(icbhi), 4),
        'beats_m2': beats_m2,
        'beats_m3': beats_m3,
    },
    'ablation': {
        'ablation_group': 'backbone_architecture',
        'ablation_role': 'variant',
        'baseline_model_id': 'M12',
        'variable_changed': 'gated_feature_fusion_ensemble_M2_plus_M3',
        'variables_held_constant': [
            'frozen_m2_backbone: 2D_CNN',
            'frozen_m3_backbone: MobileNetV2',
            'loss_function: inverse_frequency_CrossEntropyLoss',
            'seed: 42',
        ],
        'component_flags': {
            'has_sound_event_head': True,
            'has_disease_head': False,
            'has_cross_task_consistency': False,
            'has_gated_feature_fusion': True,
            'has_cqkd_regularization': False,
            'has_openmax_rejection': False,
            'owl_stage': 0,
            'compression_clusters': None,
        },
        'loss_weights': {
            'sound_event_weight': 1.0,
            'disease_weight': None,
            'consistency_weight': None,
        },
    },
    'training_history': history,
}

for out_dir in sorted({CFG['results_dir'], BASE_DIR}):
    os.makedirs(out_dir, exist_ok=True)
    rpath = os.path.join(out_dir, 'results_M30.json')
    with open(rpath, 'w') as f:
        json.dump(results, f, indent=2, default=str)
    print(f'✅ Saved: {rpath}')


✅ Saved: /kaggle/working/results_M30.json
✅ Saved: /kaggle/working/results_M30/results_M30.json


## Section 8: Bundle & Download


In [ ]:
# ============================================================
# Section 8: Bundle & Download Output Files
# ============================================================
from IPython.display import display, HTML, FileLink

zip_name = 'M30_results_bundle'
zip_path = os.path.join(BASE_DIR, zip_name)
if os.path.exists(zip_path + '.zip'): os.remove(zip_path + '.zip')

archive = shutil.make_archive(zip_path, 'zip', CFG['results_dir'])
size_mb = os.path.getsize(archive) / (1024 * 1024)

print(f"\n{'='*60}")
print('M30 RESULTS DOWNLOAD BUNDLE')
print(f"{'='*60}")
print(f'Zip: {archive} ({size_mb:.2f} MB)')

if PLATFORM == 'Kaggle':
    print('\n📥 Kaggle Clickable Download Link:')
    display(FileLink('M30_results_bundle.zip'))

try:
    with open(archive, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode('utf-8')
    href = f'data:application/zip;base64,{b64}'
    html = f'''
<div style="background:#e7f5ff;border:1px solid #74c0fc;padding:16px;border-radius:8px;margin:12px 0;">
  <h3 style="margin-top:0;color:#1864ab;">📥 M30 Results Bundle ({size_mb:.2f} MB)</h3>
  <a href="{href}" download="M30_results_bundle.zip"
     style="display:inline-block;background:#1c7ed6;color:white;padding:12px 24px;
            text-decoration:none;border-radius:6px;font-weight:bold;">⬇️ Download M30_results_bundle.zip</a>
</div>'''
    display(HTML(html))
except Exception as e:
    print(f'Download note: {e}')



M30 RESULTS DOWNLOAD BUNDLE
Zip: /kaggle/working/M30_results_bundle.zip (86.81 MB)

📥 Kaggle Clickable Download Link:


/kaggle/working/M30_results_bundle.zip